# Portion Sizing Benchmark
## Testing on Portion Estimation

**What this does:**
1. Downloads Nutrition5k sample images (professionally measured portions)
2. Runs Gemini to estimate portion sizes
3. Compares predictions to actual weights
4. Shows accuracy metrics



---
## Setup

In [ ]:
# Install dependencies
!pip install google-generativeai pandas numpy pillow tqdm requests -q
!pip install anthropic



In [ ]:
import os
import google.generativeai as genai
from anthropic import Anthropic
from openai import OpenAI
import base64  # ← ADD THIS
from pathlib import Path
from PIL import Image

os.environ["GEMINI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
gemini_model = genai.GenerativeModel("gemini-2.0-flash-lite-001")

claude_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])



In [ ]:
# ========================================
# FIXED: Nutrition5k has NO header row
# ========================================
import pandas as pd

print("📥 Loading Nutrition5k (no header version)...")

url = "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/metadata/dish_metadata_cafe1.csv"

# Read WITHOUT header, only first 7 columns
df = pd.read_csv(
    url,
    header=None,  # NO header row
    usecols=[0, 1, 2, 3, 4, 5, 6],  # Only first 7 columns
    names=['dish_id', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein', 'num_ingrs'],
    on_bad_lines='skip'
)

# Convert to proper types
df['total_mass'] = pd.to_numeric(df['total_mass'], errors='coerce')
df['total_calories'] = pd.to_numeric(df['total_calories'], errors='coerce')
df['total_fat'] = pd.to_numeric(df['total_fat'], errors='coerce')
df['total_carb'] = pd.to_numeric(df['total_carb'], errors='coerce')
df['total_protein'] = pd.to_numeric(df['total_protein'], errors='coerce')

# Remove rows with invalid data
df = df.dropna(subset=['total_mass', 'total_calories']).reset_index(drop=True)

print(f"Loaded {len(df)} dishes from Nutrition5k")
print(f"\nPortion sizes:")
print(df['total_mass'].describe())

# Filter to reasonable portions (100-500g)
df_filtered = df[
    (df["total_mass"] >= 100) &
    (df["total_mass"] <= 500)
].reset_index(drop=True)

print(f"\n📊 After filtering (100-500g): {len(df_filtered)} dishes")

# Sample 50 dishes
N_SAMPLES = 50
df_sample = df_filtered.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)

print(f"Selected {len(df_sample)} dishes for benchmark")
print(f"\nFirst few:")
df_sample.head()

In [ ]:
# ========================================
# IMPROVED: Download with better handling
# ========================================
import os
import requests
from tqdm import tqdm

print("📥 Downloading dish images...\n")

os.makedirs("nutrition5k_images", exist_ok=True)

base_url = "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset"

successful_dishes = []
failed_dishes = []

# Try to download ALL dishes from df_filtered (not just 50)
# We'll keep trying until we get 50 successful downloads
print(f"Attempting to download from {len(df_filtered)} available dishes...")
print("Will stop after getting 10 successful downloads\n")

for idx, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="Downloading"):

    # Stop if we have 50 successful downloads
    if len(successful_dishes) >= 50:
        break

    dish_id = row["dish_id"]
    img_path = f"nutrition5k_images/{dish_id}.png"

    # Skip if already downloaded
    if os.path.exists(img_path):
        successful_dishes.append(row)
        continue

    # Try to download
    url = f"{base_url}/imagery/realsense_overhead/{dish_id}/rgb.png"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        # Save image
        with open(img_path, 'wb') as f:
            f.write(response.content)

        # Success!
        successful_dishes.append(row)

    except:
        failed_dishes.append(dish_id)
        continue  # Try next dish

print(f"\n Successfully downloaded {len(successful_dishes)} images")
print(f"Failed: {len(failed_dishes)} images")

# Create new df_sample with only successful downloads
if len(successful_dishes) > 0:
    df_sample = pd.DataFrame(successful_dishes).reset_index(drop=True)
    print(f"\n Ready to benchmark {len(df_sample)} dishes")
else:
    print("\n No images downloaded !!!")

In [ ]:
# Add image paths to dataframe
df_sample["image_path"] = df_sample["dish_id"].apply(
    lambda x: f"nutrition5k_images/{x}.png"
)

# Keep only rows where image exists
df_sample["exists"] = df_sample["image_path"].apply(os.path.exists)
df_sample = df_sample[df_sample["exists"]].drop("exists", axis=1).reset_index(drop=True)

print(f"{len(df_sample)} images ready for benchmark")

---
## Define Portion Categories

In [ ]:
# Portion size categories
PORTION_CATEGORIES = {
    "very_small": (0, 50),
    "small": (50, 100),
    "medium": (100, 200),
    "large": (200, 350),
    "very_large": (350, 1000)
}

def categorize_portion(grams):
    """Convert grams to category"""
    for category, (min_g, max_g) in PORTION_CATEGORIES.items():
        if min_g <= grams < max_g:
            return category
    return "very_large"

# Add ground truth categories
df_sample["true_category"] = df_sample["total_mass"].apply(categorize_portion)

print("Category distribution:")
print(df_sample["true_category"].value_counts())

---
## Run Benchmark

In [ ]:
from PIL import Image
import base64
import re
import json

# Your prompt for portion estimation
PROMPT = """
You are analyzing a food image to estimate portion size.

This is a SINGLE 2D OVERHEAD photo with NO reference objects.

Return the estimated portion size of the entire plate based on the portion categories
below.

PORTION CATEGORIES:
- very_small: 0-50g
- small: 50-100g
- medium: 100-200g
- large: 200-350g
- very_large: 350-1000g

Return STRICT JSON ONLY:
{
  "dish_name": "brief description",
  "portion_category": "medium",
  "portion_grams_estimate": 150,
  "portion_confidence": 60,
  "estimated_calories": 200
}

No preamble, just JSON.
"""

def extract_grams_from_json(text):
    """Extract portion_grams_estimate from JSON response"""
    try:
        # Remove markdown code fences if present
        clean = re.sub(r'```json\s*|\s*```', '', text).strip()

        # Parse JSON
        data = json.loads(clean)

        # Extract the grams estimate
        if 'portion_grams_estimate' in data:
            return float(data['portion_grams_estimate'])

    except Exception as e:
        print(f"   JSON parsing error: {e}")
        # Fallback: try to find any number in the text
        numbers = re.findall(r'"portion_grams_estimate":\s*(\d+)', text)
        if numbers:
            return float(numbers[0])

    return None

def get_gemini_prediction(image_path):
    """Get portion estimate from Gemini"""
    try:
        img = Image.open(image_path)
        response = gemini_model.generate_content([PROMPT, img])
        grams = extract_grams_from_json(response.text)
        return grams
    except Exception as e:
        print(f"Gemini error: {e}")
        return None

def get_claude_prediction(image_path):
    """Get portion estimate from Claude"""
    try:
        with open(image_path, "rb") as img_file:
            img_data = base64.standard_b64encode(img_file.read()).decode("utf-8")

        media_type = "image/jpeg"
        if image_path.endswith('.png'):
            media_type = "image/png"

        message = claude_client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=200,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": media_type,
                                "data": img_data,
                            },
                        },
                        {
                            "type": "text",
                            "text": PROMPT
                        }
                    ],
                }
            ],
        )
        grams = extract_grams_from_json(message.content[0].text)
        return grams
    except Exception as e:
        print(f"Claude error: {e}")
        return None

def get_gpt4_prediction(image_path):
    """Get portion estimate from GPT-4o-mini"""
    try:
        with open(image_path, "rb") as img_file:
            img_data = base64.b64encode(img_file.read()).decode('utf-8')

        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": PROMPT
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{img_data}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=200
        )
        grams = extract_grams_from_json(response.choices[0].message.content)
        return grams
    except Exception as e:
        print(f"GPT-4o-mini error: {e}")
        return None



In [ ]:
from tqdm import tqdm
import time

# Initialize columns for each model
df_sample['gemini_pred'] = None
df_sample['claude_pred'] = None
df_sample['gpt4_pred'] = None

print("🚀 Running predictions on all models...\n")

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Processing"):
    image_path = row['image_path']

    print(f"\n Dish {idx+1}/{len(df_sample)}: {row['dish_id']}")
    print(f"   Actual weight: {row['total_mass']:.0f}g")

    # ===== GEMINI =====
    try:
        gemini_pred = get_gemini_prediction(image_path)
        df_sample.at[idx, 'gemini_pred'] = gemini_pred
        print(f"   Gemini: {gemini_pred:.0f}g" if gemini_pred else "   Gemini: ERROR")
    except Exception as e:
        print(f"   Gemini: ERROR - {e}")
    time.sleep(1)  # Rate limiting

    # ===== CLAUDE =====
    try:
        claude_pred = get_claude_prediction(image_path)
        df_sample.at[idx, 'claude_pred'] = claude_pred
        print(f"   Claude: {claude_pred:.0f}g" if claude_pred else "   Claude: ERROR")
    except Exception as e:
        print(f"   Claude: ERROR - {e}")
    time.sleep(1)

    # ===== GPT-4o-mini =====
    try:
        gpt4_pred = get_gpt4_prediction(image_path)
        df_sample.at[idx, 'gpt4_pred'] = gpt4_pred
        print(f"   GPT-4: {gpt4_pred:.0f}g" if gpt4_pred else "   GPT-4: ERROR")
    except Exception as e:
        print(f"   GPT-4: ERROR - {e}")
    time.sleep(1)

print("\nAll predictions complete!")

In [ ]:
import numpy as np

# keep only rows where all 3 models succeeded
df_valid = df_sample.dropna(subset=['gemini_pred', 'claude_pred', 'gpt4_pred']).copy()

print(f"Valid predictions: {len(df_valid)}/{len(df_sample)}\n")

# Calculate errors for each model
for model in ['gemini', 'claude', 'gpt4']:
    pred_col = f'{model}_pred'
    error_col = f'{model}_error_pct'
    abs_error_col = f'{model}_abs_error'

    # Percentage error
    df_valid[error_col] = abs((df_valid[pred_col] - df_valid['total_mass']) / df_valid['total_mass'] * 100)

    # Absolute error in grams
    df_valid[abs_error_col] = abs(df_valid[pred_col] - df_valid['total_mass'])

# Save detailed results
df_valid.to_csv("multi_model_benchmark_results.csv", index=False)
print(" Detailed results saved to: multi_model_benchmark_results.csv")

# Create summary table
summary = {
    'Model': ['Gemini 2.0 Flash Lite','Claude Haiku 3','GPT-4o-mini'],
    'Mean Error (%)': [
        df_valid['gemini_error_pct'].mean(),
        df_valid['claude_error_pct'].mean(),
        df_valid['gpt4_error_pct'].mean()

    ],
    'Median Error (%)': [
        df_valid['gemini_error_pct'].median(),
        df_valid['claude_error_pct'].median(),
        df_valid['gpt4_error_pct'].median()
    ],
    'Within ±20%': [
        (df_valid['gemini_error_pct'] <= 20).sum() / len(df_valid) * 100,
        (df_valid['claude_error_pct'] <= 20).sum() / len(df_valid) * 100,
        (df_valid['gpt4_error_pct'] <= 20).sum() / len(df_valid) * 100
    ]
}

summary_df = pd.DataFrame(summary)
summary_df.to_csv("model_comparison_summary.csv", index=False)
print(" Summary saved to: model_comparison_summary.csv")

print("\n" + "="*70)
print(summary_df.to_string(index=False))
print("="*70)

---
## Evaluate Results

In [ ]:
# Print comparative metrics
print("="*70)
print("MULTI-MODEL BENCHMARK COMPARISON")
print("="*70)
print(f"\n{'Metric':<30} {'Gemini':<15} {'Claude':<15} {'GPT-4o-mini':<15}")
print("="*70)

metrics = {
    'Mean Error (%)': lambda m: df_valid[f'{m}_error_pct'].mean(),
    'Median Error (%)': lambda m: df_valid[f'{m}_error_pct'].median(),
    'Mean Abs Error (g)': lambda m: df_valid[f'{m}_abs_error'].mean(),
    'Median Abs Error (g)': lambda m: df_valid[f'{m}_abs_error'].median(),
    'Within ±20%': lambda m: (df_valid[f'{m}_error_pct'] <= 20).sum() / len(df_valid) * 100,
    'Within ±50g': lambda m: (df_valid[f'{m}_abs_error'] <= 50).sum() / len(df_valid) * 100,
}

for metric_name, metric_func in metrics.items():
    gemini_val = metric_func('gemini')
    claude_val = metric_func('claude')
    gpt4_val = metric_func('gpt4')

    if 'Within' in metric_name:
      print(f"{metric_name:<30} {gemini_val:>13.1f}%  {claude_val:>13.1f}%  {gpt4_val:>13.1f}%")
    elif '%' in metric_name:
      print(f"{metric_name:<30} {gemini_val:>13.1f}% {claude_val:>13.1f}% {gpt4_val:>13.1f}%")
    else:
      print(f"{metric_name:<30} {gemini_val:>13.1f}g  {claude_val:>13.1f}g  {gpt4_val:>13.1f}g")

    print("="*70)

# Show best performer
print("\n BEST PERFORMERS:")
print("-"*70)
models = ['gemini', 'claude', 'gpt4'] #add claude later on here plss
model_names = {'gemini': 'Gemini 2.0 Flash Lite', 'claude': 'Claude Haiku 3', 'gpt4': 'GPT-4o-mini'} # add claude model later on

for metric in ['Mean Error (%)', 'Median Error (%)']:
    values = {m: metrics[metric](m) for m in models}
    best = min(values, key=values.get)
    print(f"   {metric:<25} → {model_names[best]} ({values[best]:.1f}%)")

for metric in ['Within ±20%', 'Within ±50g']:
    values = {m: metrics[metric](m) for m in models}
    best = max(values, key=values.get)
    print(f"   {metric:<25} → {model_names[best]} ({values[best]:.1f}%)")

print("="*70)

In [ ]:
# Define categorization function
def categorize_portion(grams):
    if grams < 50:
        return "very_small"
    elif grams < 150:
        return "small"
    elif grams < 300:
        return "medium"
    elif grams < 500:
        return "large"
    else:
        return "very_large"

# Create categorical columns for ALL models
df_valid["gemini_category"] = df_valid["gemini_pred"].apply(categorize_portion)
df_valid["claude_category"] = df_valid["claude_pred"].apply(categorize_portion)
df_valid["gpt4_category"] = df_valid["gpt4_pred"].apply(categorize_portion)
df_valid["true_category"] = df_valid["total_mass"].apply(categorize_portion)

# Then run categorical analysis for each model
for model in ['gemini', 'claude', 'gpt4']:
    df_valid['pred_category'] = df_valid[f'{model}_category']
    # ... run your categorical accuracy code ...

---
## Visualizations

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Define category order
category_order = ["very_small", "small", "medium", "large", "very_large"]

# First, create categorical columns for all models
def categorize_portion(grams):
    if grams < 50:
        return "very_small"
    elif grams < 150:
        return "small"
    elif grams < 300:
        return "medium"
    elif grams < 500:
        return "large"
    else:
        return "very_large"

df_valid["gemini_category"] = df_valid["gemini_pred"].apply(categorize_portion)
df_valid["claude_category"] = df_valid["claude_pred"].apply(categorize_portion)
df_valid["gpt4_category"] = df_valid["gpt4_pred"].apply(categorize_portion)
df_valid["true_category"] = df_valid["total_mass"].apply(categorize_portion)

# Create confusion matrix for each model
models = {
    'gemini': 'Gemini 2.0 Flash Lite',
    'claude': 'Claude Haiku 3',
    'gpt4': 'GPT-4o-mini'
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (model, title) in enumerate(models.items()):
    pred_col = f'{model}_category'

    # Only use rows with valid categories
    mask = (df_valid[pred_col].isin(category_order)) & (df_valid["true_category"].isin(category_order))
    df_cm = df_valid[mask]

    cm = confusion_matrix(
        df_cm["true_category"],
        df_cm[pred_col],
        labels=category_order
    )

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=category_order,
                yticklabels=category_order,
                ax=axes[idx])
    axes[idx].set_xlabel('Predicted Category')
    axes[idx].set_ylabel('True Category')
    axes[idx].set_title(f'{title}\nConfusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = {
    'gemini_pred': 'Gemini 2.0 Flash Lite',
    'claude_pred': 'Claude Haiku 3',
    'gpt4_pred': 'GPT-4o-mini'
}

for idx, (pred_col, title) in enumerate(models.items()):
    axes[idx].scatter(df_valid["total_mass"], df_valid[pred_col], alpha=0.5)
    axes[idx].plot([0, 500], [0, 500], 'r--', label='Perfect prediction')
    axes[idx].set_xlabel('Actual Mass (grams)')
    axes[idx].set_ylabel('Predicted Mass (grams)')
    axes[idx].set_title(f'{title}')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

models = ['gemini', 'claude', 'gpt4']
model_names = ['Gemini 2.0 Flash Lite', 'Claude Haiku 3', 'GPT-4o-mini']

for idx, (model, name) in enumerate(zip(models, model_names)):
    # Percentage error
    axes[0, idx].hist(df_valid[f"{model}_error_pct"], bins=20, edgecolor='black')
    axes[0, idx].axvline(20, color='r', linestyle='--', label='±20%')
    axes[0, idx].axvline(30, color='orange', linestyle='--', label='±30%')
    axes[0, idx].set_xlabel('Error (%)')
    axes[0, idx].set_ylabel('Frequency')
    axes[0, idx].set_title(f'{name}\nPercentage Error')
    axes[0, idx].legend()

    # Absolute error
    axes[1, idx].hist(df_valid[f"{model}_abs_error"], bins=20, edgecolor='black')
    axes[1, idx].axvline(50, color='r', linestyle='--', label='±50g')
    axes[1, idx].axvline(100, color='orange', linestyle='--', label='±100g')
    axes[1, idx].set_xlabel('Error (grams)')
    axes[1, idx].set_ylabel('Frequency')
    axes[1, idx].set_title(f'{name}\nAbsolute Error')
    axes[1, idx].legend()

plt.tight_layout()
plt.show()

---
## Sample Predictions

In [ ]:
# Show a few example predictions
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (idx, row) in enumerate(df_valid.head(6).iterrows()):
    img = Image.open(row["image_path"])
    axes[i].imshow(img)
    axes[i].axis('off')

    title = f"Actual: {row['total_mass']:.0f}g ({row['true_category']})\n"
    title += f"Gemini:  {row['gemini_pred']:.0f}g ({row['gemini_category']}) - {row['gemini_error_pct']:.0f}%\n"
    title += f"Claude:  {row['claude_pred']:.0f}g ({row['claude_category']}) - {row['claude_error_pct']:.0f}%\n"
    title += f"GPT-4:   {row['gpt4_pred']:.0f}g ({row['gpt4_category']}) - {row['gpt4_error_pct']:.0f}%"

    axes[i].set_title(title, fontsize=9)

plt.tight_layout()
plt.show()

---
## Export Results

In [ ]:
# Download results CSV
from google.colab import files

# Save full results
df_valid.to_csv("nutrition5k_benchmark_results.csv", index=False)
files.download("nutrition5k_benchmark_results.csv")

print("Results downloaded!")